# 06 - LightGBM Hyperparameter Tuning

This notebook tunes the LightGBM model after establishing the default baseline and threshold-tuned result.

The existing processed train/test split is reused. No preprocessing is redone, and the held-out test set is only used for final evaluation.

## 1. Tuning Strategy

The aim is to improve LightGBM while keeping the implementation readable:

- use an internal validation split from the existing training data
- tune hyperparameters with cross-validated ROC-AUC on the internal training portion
- select a recall-focused threshold on the validation portion
- retrain the tuned model on the full existing training set
- evaluate once on the original test set

ROC-AUC is used for hyperparameter search because threshold tuning handles the medical recall tradeoff separately.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.evaluate import evaluate_scores_at_threshold, plot_roc_curve, write_results_markdown

In [ ]:
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_DATA_DIR

## 2. Load the Existing Processed Data

The processed files are loaded directly from `data/processed/`. This preserves the same train/test split used by all previous model comparisons.

In [ ]:
X_train = pd.read_csv(PROCESSED_DATA_DIR / "X_train.csv")
X_test = pd.read_csv(PROCESSED_DATA_DIR / "X_test.csv")
y_train = pd.read_csv(PROCESSED_DATA_DIR / "y_train.csv").squeeze("columns")
y_test = pd.read_csv(PROCESSED_DATA_DIR / "y_test.csv").squeeze("columns")

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
assert X_train.isna().sum().sum() == 0
assert X_test.isna().sum().sum() == 0
assert y_train.isna().sum() == 0
assert y_test.isna().sum() == 0
assert X_train.columns.equals(X_test.columns)
assert X_train.select_dtypes(exclude="number").empty
assert X_test.select_dtypes(exclude="number").empty

pd.DataFrame(
    {
        "y_train_count": y_train.value_counts().sort_index(),
        "y_train_proportion": y_train.value_counts(normalize=True).sort_index(),
        "y_test_count": y_test.value_counts().sort_index(),
        "y_test_proportion": y_test.value_counts(normalize=True).sort_index(),
    }
)

## 3. Internal Split for Tuning

The original test set is kept untouched. Hyperparameters are searched on the internal training portion, and the classification threshold is selected on the validation portion.

In [ ]:
X_model_train, X_valid, y_model_train, y_valid = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train,
)

print(f"X_model_train shape: {X_model_train.shape}")
print(f"X_valid shape: {X_valid.shape}")

## 4. Randomized Hyperparameter Search

This is a small randomized search rather than an exhaustive grid. It focuses on common LightGBM controls for tree size, learning rate, regularization, and row/column sampling.

In [ ]:
base_model = LGBMClassifier(
    objective="binary",
    random_state=42,
    verbosity=-1,
    n_jobs=1,
    subsample_freq=1,
)

param_distributions = {
    "n_estimators": [100, 150, 200, 300],
    "learning_rate": [0.01, 0.03, 0.05, 0.08, 0.10],
    "num_leaves": [7, 15, 31, 63],
    "max_depth": [-1, 3, 5, 7],
    "min_child_samples": [10, 20, 30, 50],
    "subsample": [0.8, 0.9, 1.0],
    "colsample_bytree": [0.8, 0.9, 1.0],
    "reg_alpha": [0.0, 0.1, 0.5, 1.0],
    "reg_lambda": [0.0, 0.1, 0.5, 1.0],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_distributions,
    n_iter=40,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    random_state=42,
    refit=True,
)

search.fit(X_model_train, y_model_train)

print(f"Best validation CV ROC-AUC: {search.best_score_:.3f}")
search.best_params_

In [ ]:
cv_results_df = pd.DataFrame(search.cv_results_).sort_values(
    "rank_test_score"
).reset_index(drop=True)

top_search_results = cv_results_df[
    ["rank_test_score", "mean_test_score", "std_test_score", "params"]
].head(10)

top_search_results

In [ ]:
top_plot_df = cv_results_df.head(10).sort_values("mean_test_score")

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(range(len(top_plot_df)), top_plot_df["mean_test_score"], color="#4C78A8")
ax.set_yticks(range(len(top_plot_df)))
ax.set_yticklabels([f"Rank {rank}" for rank in top_plot_df["rank_test_score"]])
ax.set_xlabel("Mean CV ROC-AUC")
ax.set_title("Top LightGBM Hyperparameter Search Results")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "lightgbm_hyperparameter_search.png", dpi=150, bbox_inches="tight")

## 5. Tune the Threshold for the Tuned Model

After selecting hyperparameters, the tuned model is fitted on the internal training portion and scored on the validation portion. The threshold selection rule is the same as before: prefer validation recall of at least `0.95`, then choose the best F1 score.

In [ ]:
best_params = search.best_params_

validation_model = LGBMClassifier(
    objective="binary",
    random_state=42,
    verbosity=-1,
    n_jobs=-1,
    subsample_freq=1,
    **best_params,
)
validation_model.fit(X_model_train, y_model_train)

valid_proba = validation_model.predict_proba(X_valid)[:, 1]
thresholds = np.round(np.arange(0.10, 0.81, 0.01), 2)

validation_threshold_df = pd.DataFrame(
    [
        evaluate_scores_at_threshold("LightGBM tuned validation", y_valid, valid_proba, threshold)
        for threshold in thresholds
    ]
)

candidate_thresholds = validation_threshold_df[validation_threshold_df["Recall"] >= 0.95]

if not candidate_thresholds.empty:
    selected_threshold_row = candidate_thresholds.sort_values(
        ["F1", "Precision", "Threshold"],
        ascending=[False, False, False],
    ).iloc[0]
else:
    selected_threshold_row = validation_threshold_df.sort_values(
        ["Recall", "F1", "Precision"],
        ascending=[False, False, False],
    ).iloc[0]

selected_threshold = float(selected_threshold_row["Threshold"])
selected_threshold_row

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(validation_threshold_df["Threshold"], validation_threshold_df["Precision"], label="Precision")
ax1.plot(validation_threshold_df["Threshold"], validation_threshold_df["Recall"], label="Recall")
ax1.plot(validation_threshold_df["Threshold"], validation_threshold_df["F1"], label="F1")
ax1.axvline(selected_threshold, color="black", linestyle="--", label=f"Selected threshold={selected_threshold:.2f}")
ax1.set_xlabel("Threshold")
ax1.set_ylabel("Score")
ax1.set_ylim(0, 1.05)
ax1.grid(alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(
    validation_threshold_df["Threshold"],
    validation_threshold_df["False Negatives"],
    color="tab:red",
    linestyle=":",
    label="False Negatives",
)
ax2.set_ylabel("False Negatives")

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="center right")
ax1.set_title("Tuned LightGBM Validation Threshold Tradeoff")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "lightgbm_tuned_params_threshold_tradeoff.png", dpi=150, bbox_inches="tight")

## 6. Final Test Evaluation

The tuned model is retrained on the full existing training set. The test set is evaluated with both the default `0.50` threshold and the selected validation threshold.

In [ ]:
final_model = LGBMClassifier(
    objective="binary",
    random_state=42,
    verbosity=-1,
    n_jobs=-1,
    subsample_freq=1,
    **best_params,
)
final_model.fit(X_train, y_train)

test_proba = final_model.predict_proba(X_test)[:, 1]

tuned_params_default_result = evaluate_scores_at_threshold(
    "LightGBM tuned params",
    y_test,
    test_proba,
    threshold=0.50,
)

tuned_params_threshold_result = evaluate_scores_at_threshold(
    "LightGBM tuned params + threshold",
    y_test,
    test_proba,
    threshold=selected_threshold,
)

pd.DataFrame([tuned_params_default_result, tuned_params_threshold_result])

## 7. Save Evaluation Figures

The confusion matrices show the effect of the default and selected thresholds. The ROC curve remains threshold-independent and reflects probability ranking.

In [ ]:
threshold_results = [
    ("Default threshold", 0.50, tuned_params_default_result),
    ("Selected threshold", selected_threshold, tuned_params_threshold_result),
]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, (title, threshold, result) in zip(axes, threshold_results):
    y_pred = (test_proba >= threshold).astype(int)
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    ax.imshow(cm, interpolation="nearest", cmap="Blues")
    ax.set_title(f"{title}\nThreshold={threshold:.2f}, Recall={result['Recall']:.3f}, FN={result['False Negatives']}")
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["No Heart Disease", "Heart Disease"], rotation=30, ha="right")
    ax.set_yticklabels(["No Heart Disease", "Heart Disease"])

    labels = [["TN", "FP"], ["FN", "TP"]]
    threshold_for_text = cm.max() / 2
    for row in range(2):
        for col in range(2):
            color = "white" if cm[row, col] > threshold_for_text else "black"
            ax.text(col, row, f"{labels[row][col]}\n{cm[row, col]}", ha="center", va="center", color=color, fontsize=11)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "lightgbm_tuned_params_confusion_matrices.png", dpi=150, bbox_inches="tight")

roc_fig, _ = plot_roc_curve(
    {"LightGBM tuned params": final_model},
    X_test,
    y_test,
    save_path=FIGURES_DIR / "lightgbm_tuned_params_roc_curve.png",
    title="LightGBM Tuned Params ROC Curve",
)

In [ ]:
feature_importance_df = pd.DataFrame(
    {
        "Feature": X_train.columns,
        "Importance": final_model.feature_importances_,
    }
).sort_values("Importance", ascending=False)

top_features = feature_importance_df.head(15).sort_values("Importance")

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top_features["Feature"], top_features["Importance"], color="#4C78A8")
ax.set_title("Tuned LightGBM Feature Importance")
ax.set_xlabel("Importance")
ax.set_ylabel("Feature")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "lightgbm_tuned_params_feature_importance.png", dpi=150, bbox_inches="tight")

feature_importance_df.head(15)

## 8. Update the Results Report

The tuned-parameter rows are appended to the existing comparison table. The previous rows are kept so the effect of tuning is visible.

In [ ]:
def read_results_table(markdown_path):
    table_lines = [
        line.strip()
        for line in markdown_path.read_text(encoding="utf-8").splitlines()
        if line.strip().startswith("|")
    ]

    rows = []
    header = None

    for line in table_lines:
        values = [value.strip() for value in line.strip("|").split("|")]
        if all(set(value) <= {"-", ":"} for value in values):
            continue
        if header is None:
            header = values
        else:
            rows.append(values)

    results = pd.DataFrame(rows, columns=header)
    for column in results.columns:
        if column != "Model":
            results[column] = pd.to_numeric(results[column])

    return results


existing_model_names = [
    "Logistic Regression",
    "Random Forest",
    "XGBoost",
    "LightGBM",
    "LightGBM tuned threshold",
]

existing_results_df = read_results_table(REPORTS_DIR / "results.md")
existing_results_df = existing_results_df[
    existing_results_df["Model"].isin(existing_model_names)
].copy()

new_results_df = pd.DataFrame([tuned_params_default_result, tuned_params_threshold_result])

comparison_columns = [
    "Model",
    "Threshold",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC",
    "True Negatives",
    "False Positives",
    "False Negatives",
    "True Positives",
]

comparison_df = pd.concat(
    [existing_results_df[comparison_columns], new_results_df[comparison_columns]],
    ignore_index=True,
)

write_results_markdown(
    comparison_df,
    REPORTS_DIR / "results.md",
    title="Model Comparison Results",
    model_group_name="baseline models, LightGBM threshold tuning, and LightGBM hyperparameter tuning",
    best_model_label="model",
)

comparison_df

## 9. Summary

Hyperparameter tuning improves the default-threshold LightGBM result by reducing false negatives and increasing recall.

Combining tuned hyperparameters with the validation-selected threshold keeps false negatives very low while producing fewer false positives than the earlier threshold-only LightGBM approach.

This is still a modeling experiment, not a clinical decision rule. SHAP interpretation remains a future step.